# 01 · Подготовка HierText
Из внешнего датасета получаем правильно ориентированные слова. Повороты 0/180°
создаются позже в загрузчике, без повторного JPEG-сжатия и интерполяции.

Важное ограничение первого варианта: берём читаемые не вертикальные слова
с четырьмя вершинами. Искривлённые полигоны пропускаем и считаем в отчёте.
Это пока не полное покрытие домена Авито и не набор строк.

Ориентацию сохраняем по официальному порядку вершин:
https://github.com/google-research-datasets/hiertext/blob/main/docs/images/vertices.png
Источник и лицензия: https://github.com/google-research-datasets/hiertext (CC BY-SA 4.0).


In [ ]:
from pathlib import Path


def find_project_root(start=None):
    from pathlib import Path
    import os

    current = Path(start or Path.cwd()).resolve()
    explicit = os.environ.get("AVITO_PROJECT_ROOT")
    candidates = (
        [Path(explicit).expanduser().resolve()] if explicit else [current, *current.parents]
    )
    for candidate in candidates:
        marker = (candidate / "avito_project.json").is_file()
        layout = (candidate / "notebooks" / "_shared.ipynb").is_file() and (
            candidate / "requirements.txt"
        ).is_file()
        if marker or layout:
            return candidate
    raise RuntimeError(
        f"Не найден корень Avito_test. Текущая папка: {current}. "
        "Перенеси notebooks/ целиком, requirements.txt и avito_project.json в папку проекта. "
        "На VM выполни в отдельной ячейке: %cd /home/slomauh/ml/Avito_test . "
        "Для другого расположения задай переменную окружения AVITO_PROJECT_ROOT. "
        + (f"Сейчас AVITO_PROJECT_ROOT={explicit!r}; проверь этот путь." if explicit else "")
    )


ROOT = find_project_root()
get_ipython().run_line_magic("run", str(ROOT / "notebooks" / "_shared.ipynb"))
print("Проект:", ROOT)


In [ ]:
RAW_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "data/processed/hiertext_v1"
DOWNLOAD = False  # True, если нужно скачать ~3.5 ГБ на сервер
EXTRACT = False  # True после скачивания архивов
MAX_IMAGES = None  # Для первой проверки: 50; для обучения: None
# Для пробного и полного датасетов используй разные OUTPUT_DIR.

# RAW_DIR/train и RAW_DIR/validation должны содержать HierText, а RAW_DIR/gt — его разметку.


In [ ]:
import gzip, tarfile, urllib.request
from collections import Counter
import cv2
import ijson


def download(url, target):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        print("Уже существует:", target.name)
        return
    partial = target.with_suffix(target.suffix + ".part")
    with urllib.request.urlopen(url, timeout=60) as response, open(partial, "wb") as out:
        total = int(response.headers.get("Content-Length", 0))
        with tqdm(total=total or None, unit="B", unit_scale=True, desc=target.name) as bar:
            for chunk in iter(lambda: response.read(1024 * 1024), b""):
                out.write(chunk)
                bar.update(len(chunk))
    if total and partial.stat().st_size != total:
        raise IOError("Неполная загрузка; повтори ячейку")
    partial.replace(target)


def safe_extract(archive, target):
    target = Path(target).resolve()
    target.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall(target, filter="data")  # Python >=3.11.8/3.12


def records(path):
    # Несмотря на расширение .jsonl, официальная разметка — JSON с annotations[].
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rb") as f:
        yield from ijson.items(f, "annotations.item", use_float=True)


def rectify_word(image, vertices, margin=0.03):
    pts = np.asarray(vertices, dtype=np.float32)
    if pts.shape != (4, 2) or not np.isfinite(pts).all():
        raise ValueError("Для baseline используем только четырёхугольные слова")
    if not cv2.isContourConvex(pts.reshape(-1, 1, 2)) or abs(cv2.contourArea(pts)) < 4:
        raise ValueError("Вырожденный четырёхугольник")
    # Порядок HierText: TL, TR, BR, BL относительно текста, а не изображения!
    # Не применять сортировку по x/y или minAreaRect: это потеряет 180° ориентацию.
    w = max(np.linalg.norm(pts[1] - pts[0]), np.linalg.norm(pts[2] - pts[3]))
    h = max(np.linalg.norm(pts[3] - pts[0]), np.linalg.norm(pts[2] - pts[1]))
    if min(w, h) < 6 or w / h > 40:
        raise ValueError("Слишком маленький или длинный кроп")
    pts = (pts - pts.mean(0)) * (1 + 2 * margin) + pts.mean(0)
    w, h = int(round(w * (1 + 2 * margin))), int(round(h * (1 + 2 * margin)))
    scale = min(1.0, 2048 / max(w, h))
    w, h = max(2, round(w * scale)), max(2, round(h * scale))
    dst = np.array([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]], np.float32)
    transform = cv2.getPerspectiveTransform(pts, dst)
    return cv2.warpPerspective(image, transform, (w, h), borderMode=cv2.BORDER_REPLICATE)


def assign_split(group_id, official_split, seed=42):
    value = int(hashlib.sha256(f"{seed}:{group_id}".encode()).hexdigest()[:8], 16) / 2**32
    if official_split == "train":
        return "dev" if value < 0.1 else "train"
    return "calibration" if value < 0.5 else "holdout"


def prepare_hiertext(raw_dir, output_dir, max_images=None, seed=42):
    raw_dir, output_dir = Path(raw_dir), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    rows, counters, provenance = [], Counter(), {}
    for official in ["train", "validation"]:
        ann = raw_dir / "gt" / f"{official}.jsonl.gz"
        if not ann.exists():
            ann = raw_dir / "gt" / f"{official}.jsonl"
        if not ann.exists():
            raise FileNotFoundError(f"Нет разметки: {ann}")
        provenance[official] = {"annotations_sha256": sha256_file(ann)}
        count = 0
        for item in tqdm(records(ann), desc=official):
            if max_images is not None and count >= max_images:
                break
            count += 1
            source_id = str(item["image_id"])
            if Path(source_id).name != source_id:
                raise ValueError("Неверный image_id в разметке")
            path = (
                raw_dir / official / (source_id if Path(source_id).suffix else source_id + ".jpg")
            )
            if not path.exists():
                raise FileNotFoundError(path)
            im = np.asarray(open_rgb(path))
            if im.shape[:2] != (int(item["image_height"]), int(item["image_width"])):
                raise ValueError(f"Размер отличается от разметки: {source_id}")
            group_id = sha256_file(path)
            split = assign_split(group_id, official, seed)
            for pi, paragraph in enumerate(item["paragraphs"]):
                for li, line in enumerate(paragraph["lines"]):
                    for wi, word in enumerate(line["words"]):
                        if (
                            not word.get("legible", False)
                            or word.get("vertical", False)
                            or not word.get("text", "").strip()
                        ):
                            counters["unreadable_or_vertical"] += 1
                            continue
                        try:
                            crop = rectify_word(im, word["vertices"])
                        except ValueError:
                            counters["geometry_filtered"] += 1
                            continue
                        crop_id = f"ht_{official}_{source_id}_{pi}_{li}_{wi}"
                        dest = output_dir / "crops" / official / source_id / f"{crop_id}.png"
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        Image.fromarray(crop).save(dest)
                        pixel_hash = hashlib.sha256(
                            str(crop.shape).encode() + crop.tobytes()
                        ).hexdigest()
                        rows.append(
                            {
                                "crop_id": crop_id,
                                "source_id": source_id,
                                "group_id": group_id,
                                "source": "hiertext",
                                "official_split": official,
                                "split": split,
                                "path": dest.relative_to(ROOT).as_posix(),
                                "text": word["text"],
                                "script": script_of(word["text"]),
                                "width": crop.shape[1],
                                "height": crop.shape[0],
                                "pixel_hash": pixel_hash,
                            }
                        )
        if count == 0:
            raise ValueError(f"Не прочитано ни одной записи: {ann}")
        provenance[official]["images_processed"] = count
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("Нет пригодных кропов")
    # Если одинаковая фотография встретилась в официальных train и validation,
    # оставляем только validation. Для одинаковых кропов приоритет также у отложенных.
    val_groups = set(df.loc[df.official_split == "validation", "group_id"])
    keep = ~((df.official_split == "train") & df.group_id.isin(val_groups))
    counters["duplicate_source_removed"] = int((~keep).sum())
    df = df.loc[keep].copy()
    priority = {"holdout": 0, "calibration": 1, "dev": 2, "train": 3}
    df["_priority"] = df.split.map(priority)
    old_len = len(df)
    df = (
        df.sort_values(["_priority", "crop_id"])
        .drop_duplicates("pixel_hash")
        .drop(columns="_priority")
    )
    counters["duplicate_crops_removed"] = old_len - len(df)
    df = df.sort_values("crop_id").reset_index(drop=True)
    df.to_csv(output_dir / "manifest.csv", index=False)
    report = {
        "seed": seed,
        "max_images_per_official_split": max_images,
        "provenance": provenance,
        "filtered": dict(counters),
        "counts": df.split.value_counts().to_dict(),
        "manifest_sha256": sha256_file(output_dir / "manifest.csv"),
        "orientation": "HierText ordered quadrilateral -> upright; labels generated later",
        "limitations": "Only legible, nonvertical, quadrilateral WORD crops; no curved words or line crops.",
    }
    save_json(output_dir / "preparation.json", report)
    return df, report


In [ ]:
if DOWNLOAD:
    for split in ["train", "validation"]:
        download(
            f"https://open-images-dataset.s3.amazonaws.com/ocr/{split}.tgz",
            RAW_DIR / f"{split}.tgz",
        )
        download(
            f"https://raw.githubusercontent.com/google-research-datasets/hiertext/main/gt/{split}.jsonl.gz",
            RAW_DIR / "gt" / f"{split}.jsonl.gz",
        )
if EXTRACT:
    for split in ["train", "validation"]:
        if (RAW_DIR / split).exists():
            print("Папка уже существует, распаковка пропущена:", split)
        else:
            safe_extract(RAW_DIR / f"{split}.tgz", RAW_DIR)


In [ ]:
if (OUTPUT_DIR / "manifest.csv").exists():
    raise FileExistsError(
        "Manifest уже существует. Для повторной подготовки выбери новое OUTPUT_DIR."
    )
manifest, report = prepare_hiertext(RAW_DIR, OUTPUT_DIR, max_images=MAX_IMAGES)
manifest = load_manifest(OUTPUT_DIR / "manifest.csv")
display(manifest.groupby(["split", "script"]).size().unstack(fill_value=0))
print(json.dumps(report, ensure_ascii=False, indent=2))


Разбиение фиксировано по исходным фотографиям:
- официальный train → train (~90%) и dev (~10%) для выбора эпохи;
- официальный validation → calibration (~50%) и holdout (~50%).
На calibration подбираем только температуру; на holdout оцениваем уже выбранный вариант.
Официальный test HierText не нужен. Тест Авито здесь вообще не читается.


In [ ]:
import matplotlib.pyplot as plt

preview = manifest[manifest.split == "train"].sample(
    min(12, sum(manifest.split == "train")), random_state=SEED
)
fig, axes = plt.subplots(4, 3, figsize=(13, 8))
for ax, (_, row) in zip(axes.flat, preview.iterrows()):
    ax.imshow(open_rgb(ROOT / row.path))
    ax.set_title(row.text[:35], fontsize=9)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
# Это внешние обучающие данные: визуальная проверка допустима и нужна.
